# 17 — Guardrails & Deterministic Policy Enforcement

## Learning requirements
- guardrail != complete security boundary;
- input/output/model/tool/memory guardrails có responsibility khác nhau;
- deterministic policy phải bảo vệ authorization và irreversible actions;
- model-based classifier chỉ hỗ trợ nơi semantic judgment thực sự cần thiết;
- hiểu fail-open, fail-closed, abstain và escalation.

## Defense pipeline
```text
User Input
   -> Input Policy
   -> Agent / Model
   -> Tool Proposal
   -> Tool Policy + Authorization
   -> HITL when required
   -> Tool Execution
   -> Tool Result Policy
   -> Output Policy
   -> User
```

## Where guardrails belong

| Layer | Typical checks |
|---|---|
| Input | size, PII, prohibited input types, tenant context |
| Model call | context budget, allowed model, instruction hierarchy |
| Tool proposal | tool allow-list, schema, authorization, risk class |
| Tool result | untrusted-content labeling, size, secret filtering |
| Memory write | provenance, scope, retention, poison checks |
| Final output | schema, sensitive-data policy, citation/evidence rules |

Một output filter không thể sửa việc agent đã gửi email hoặc xóa record sai.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class Decision(str, Enum):
    ALLOW = "allow"
    DENY = "deny"
    REQUIRE_APPROVAL = "require_approval"

@dataclass(frozen=True)
class ToolRequest:
    tool: str
    action: str
    user_role: str
    target_owner: str
    user_id: str

READ_ONLY = {"search_docs", "get_project"}
PRIVILEGED = {"send_email", "delete_project", "publish_spec"}

def evaluate_tool_policy(req: ToolRequest) -> Decision:
    if req.tool in READ_ONLY and req.target_owner == req.user_id:
        return Decision.ALLOW
    if req.tool in PRIVILEGED and req.user_role == "admin":
        return Decision.REQUIRE_APPROVAL
    return Decision.DENY

tests = [
    ToolRequest("get_project", "read", "viewer", "u1", "u1"),
    ToolRequest("delete_project", "delete", "viewer", "u1", "u1"),
    ToolRequest("delete_project", "delete", "admin", "u1", "u1"),
]
for t in tests:
    print(t.tool, evaluate_tool_policy(t))

## LangChain middleware investigation

Đọc current middleware APIs và thực hành ít nhất:
- Human-in-the-loop middleware cho privileged tools;
- model/tool call limits để chống unbounded loops;
- PII middleware;
- fallback/retry middleware;
- một custom middleware tự enforce policy trước tool execution.

HITL decision cần xem **exact proposed action** và nên hỗ trợ approve/edit/reject khi phù hợp.

## Exercise — Policy matrix

Tạo policy matrix cho ít nhất 10 capabilities của capstone:
- read source;
- search index;
- write project memory;
- modify spec;
- send email;
- execute command;
- access external URL;
- create MCP session;
- delete artifact;
- publish final specification.

Với mỗi capability define:
principal, resource scope, allow/deny, approval, validation, audit event và failure mode.

## Required output
- `artifacts/security/policy-matrix.md`
- automated policy tests;
- one LangChain middleware lab.

## Done criteria
- Authorization không phụ thuộc vào model output.
- Unknown tool/action mặc định deny.
- Irreversible actions không silently fail-open.
- Guardrail failure behavior được define explicit.